# 🛡️ DSCAD 기반 Dilithium 부채널 공격: 통합 데이터 전처리

이 노트북은 가이드라인의 모든 핵심 전략을 통합하여 최상의 신호를 추출합니다.

## 📋 주요 파이프라인
1. **Robust HW 계산**: 32비트 정수 비트 패턴 기준 (음수 및 오버플로우 대응)
2. **Signal Purifying**: Moving Average Filter (Window: 5)로 고주파 노이즈 제거
3. **DPT (Data Power Trace)**: 전체 평균 감산을 통한 데이터 의존 신호 분리
4. **Ontology POI**: 상관계수 Peak(250개) 중심의 시맨틱 특징 추출
5. **Statistical Consistency**: 훈련 단계의 모든 통계량(Smooth Mean, Std) 저장

In [32]:
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
from scipy.signal import find_peaks

# 1. 경로 및 파라미터 설정
DATASET_DIR = '../Dataset'
RHO_PATH = '../Pearson correlation coefficient/rho.npy'
OUTPUT_DIR = './'

def get_hw(n):
    # 🚀 32비트 부호 없는 정수 비트 패턴 기준 (음수 대응 핵심)
    return bin(int(n) & 0xFFFFFFFF).count('1')

def moving_average(data, window_size=5):
    kernel = np.ones(window_size) / window_size
    if len(data.shape) == 1:
        return np.convolve(data, kernel, mode='same')
    return np.apply_along_axis(lambda m: np.convolve(m, kernel, mode='same'), axis=1, arr=data)

train_files = [os.path.join(DATASET_DIR, f'profiling_40000traces_set{i}.npy') for i in range(1, 5)]
print("✅ 환경 설정 및 유틸리티 정의 완료")

✅ 환경 설정 및 유틸리티 정의 완료


## 2. 전역 통계량 및 POI 식별

DPT 가공을 위한 전체 평균과 상관계수 기반 Peak 지점을 찾습니다.

In [33]:
print("📊 1단계: Global Mean Trace 계산 중...")
sum_trace = np.zeros(40000, dtype=np.float64)
total_count = 0
for path in train_files:
    traces = np.load(path, mmap_mode='r')
    sum_trace += np.sum(traces, axis=0)
    total_count += len(traces)
global_mean_raw = sum_trace / total_count

print("📊 2단계: 온톨로지 기반 POI(Peak) 탐색 중...")
rho = np.load(RHO_PATH)
target_rho = np.abs(rho[0]) # Poly 0, Coeff 50 기준
peaks, properties = find_peaks(target_rho, height=0.03, distance=30)
top_peak_indices = peaks[np.argsort(properties['peak_heights'])[-250:]] # 상위 250개 Peak

expanded_indices = []
for idx in top_peak_indices:
    for w in range(-7, 8): # Window Size: 15
        expanded_indices.append(idx + w)
final_poi_indices = np.sort(np.clip(np.unique(expanded_indices), 0, 39999))

print(f"✅ {len(final_poi_indices)}개의 특징 지점 식별 완료")

📊 1단계: Global Mean Trace 계산 중...
📊 2단계: 온톨로지 기반 POI(Peak) 탐색 중...
✅ 60개의 특징 지점 식별 완료


## 3. 통합 전처리 루프 (Filter + DPT + Norm)

모든 기법을 적용하여 최종 학습용 데이터를 생성합니다.

In [34]:
print("📊 3단계: 훈련 데이터 통합 가공 중...")
all_poi_traces = []

# 훈련 시 기준이 될 Smooth Mean 계산
mean_smooth = moving_average(global_mean_raw, window_size=5)

# 표준편차 추정 (DPT 상태 기준)
sample_traces = np.load(train_files[0], mmap_mode='r')[:5000]
sample_smooth = moving_average(sample_traces, window_size=5)
std_dpt = np.std(sample_smooth - mean_smooth, axis=0)
std_dpt[std_dpt == 0] = 1.0

for path in train_files:
    traces = np.load(path, mmap_mode='r')
    # 1) Filter
    traces_smooth = moving_average(traces, window_size=5)
    # 2) DPT Subtraction & 3) Vertical Normalization
    norm_traces = (traces_smooth[:, final_poi_indices] - mean_smooth[final_poi_indices]) / std_dpt[final_poi_indices]
    all_poi_traces.append(norm_traces.astype(np.float32))
    print(f"✅ {os.path.basename(path)} 가공 완료")

X_train = np.vstack(all_poi_traces)

print(f"📊 4단계: Robust HW 라벨 생성 중...")
u_labels = np.load(os.path.join(DATASET_DIR, 'profiling_40000_u=cs.npy'))
y_hw = np.array([get_hw(val) for val in u_labels[:, 0, 50]])

print(f"🚀 최종 데이터 크기 확인: X {X_train.shape}, y {y_hw.shape}")

📊 3단계: 훈련 데이터 통합 가공 중...
✅ profiling_40000traces_set1.npy 가공 완료
✅ profiling_40000traces_set2.npy 가공 완료
✅ profiling_40000traces_set3.npy 가공 완료
✅ profiling_40000traces_set4.npy 가공 완료
📊 4단계: Robust HW 라벨 생성 중...
🚀 최종 데이터 크기 확인: X (40000, 60), y (40000,)


## 4. 최종 데이터 및 통계량 저장

공격 평가 시 동일한 기준을 적용하기 위해 사용된 모든 통계량을 저장합니다.

In [35]:
np.save('X_train_poi400.npy', X_train)
np.save('y_train_hw.npy', y_hw)
np.save('poi_indices.npy', final_poi_indices)
np.save('used_mean.npy', mean_smooth) # 필터링된 평균
np.save('used_std.npy', std_dpt)     # DPT 기준 표준편차

print(f"💾 모든 파일이 저장되었습니다. 이제 모델 학습을 시작하세요!")

💾 모든 파일이 저장되었습니다. 이제 모델 학습을 시작하세요!
